In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

NPZ_PATHS = ["../data/chess_data_snipped.npz"]
BATCH_SIZE = 256
VAL_FRAC   = 0.1
CKPT_PATH  = "../weights/

in_ch = 17

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
#-----------------------------
# Combine + create data set funcs/classes

import numpy as np
import torch
from torch.utils.data import Dataset, ConcatDataset

class ChessEvalNPZ(Dataset):
    def __init__(self, npz_path: str, flatten: bool = True, max_abs_cp: float = 13000.0):
        self.path = npz_path
        self.flatten = flatten
        with np.load(self.path, mmap_mode='r') as zf:
            Zcp = zf['evaluations']
            self.N = int(Zcp.shape[0])
            self.M_cp = float(max_abs_cp)
        self._X = None; self._Zcp = None

    def _ensure_open(self):
        if self._X is None:
            zf = np.load(self.path, mmap_mode='r')
            self._X   = zf['positions']      # 
            self._Zcp = zf['evaluations']    # (N,)        memmap

    def __len__(self): 
        return self.N

    def __getitem__(self, idx):
        self._ensure_open()
        x = self._X[idx].astype(np.float32)        
        if self.flatten:
            x = x.reshape(-1)
        z_cp = float(self._Zcp[idx])

        # keep your clip at +-1500 (global_M)
        cp = np.clip(z_cp, -self.M_cp, self.M_cp)

        # train on bounded value in [-1, 1]
        y = np.tanh(cp / 400.0)

        return torch.from_numpy(x), torch.tensor([y], dtype=torch.float32)




In [3]:
# -----------------------------
# Building loaders



global_M = 1500.0
datasets = [ChessEvalNPZ(p, flatten=False, max_abs_cp=global_M) for p in NPZ_PATHS]
full_ds = ConcatDataset(datasets)
N = len(full_ds)
val_len = int(N * VAL_FRAC)
train_len = N - val_len

g = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full_ds, [train_len, val_len], generator=g)

NUM_WORKERS = 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Dataset: N={N} | input_dim={in_ch} | train={train_len} | val={val_len}")
xb0, yb0 = next(iter(train_loader))
print("One sample check:", xb0.shape)  # (B, 796)

Dataset: N=16057671 | input_dim=17 | train=14451904 | val=1605767
One sample check: torch.Size([256, 17, 8, 8])


In [4]:
xb0, yb0 = next(iter(train_loader))
print("One sample check:", xb0.shape)
assert xb0.shape[1] == in_ch, f"Expected {in_ch} channels, got {xb0.shape[1]}"


One sample check: torch.Size([256, 17, 8, 8])


In [5]:
# a) shapes & dtypes
print("xb0:", xb0.shape, xb0.dtype, "yb0:", yb0.shape, yb0.dtype)

# b) value ranges
xb_min, xb_max = float(xb0.min()), float(xb0.max())
print(f"X range first batch: [{xb_min}, {xb_max}]")

# c) targets in [0,1] after transform?
print("y batch stats: min=", float(yb0.min()), "max=", float(yb0.max()))

# d) no NaNs/inf in a couple random batches
def has_bad(t): 
    return torch.isnan(t).any().item() or torch.isinf(t).any().item()

for i, (xchk, ychk) in enumerate(train_loader):
    if i == 3: break
    assert not has_bad(xchk), "NaN/Inf in inputs"
    assert not has_bad(ychk), "NaN/Inf in targets"
print("Basic data checks passed.")


xb0: torch.Size([256, 17, 8, 8]) torch.float32 yb0: torch.Size([256, 1]) torch.float32
X range first batch: [-1.0, 1.0]
y batch stats: min= -0.998894453048706 max= 0.998894453048706
Basic data checks passed.


In [6]:

import random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F

EPOCHS   = 60          # cosine schedule works best with a fixed horizon
PATIENCE = 10          # early stop still fine

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# ResNet-ish trunk hyperparams (good starting point)
TRUNK_CH = 192
N_BLOCKS = 8
FC_HIDDEN = (256,)
P_DROP = 0.10

LR = 3e-4
WEIGHT_DECAY = 3e-5     # usually better than 1e-4 here
MAX_GRAD_NORM = 1.0




In [7]:

# -----------------------------
# Model: small ResNet for 8x8
# -----------------------------
class ResBlock(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(ch)

    def forward(self, x):
        y = self.conv1(x)
        y = self.bn1(y)
        y = F.relu(y, inplace=True)
        y = self.conv2(y)
        y = self.bn2(y)
        return F.relu(x + y, inplace=True)

class CNN(nn.Module):
    def __init__(self, in_ch: int, trunk_ch: int = 192, n_blocks: int = 8, fc_hidden=(256,), p_drop=0.1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, trunk_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(trunk_ch),
            nn.ReLU(inplace=True),
        )
        self.blocks = nn.Sequential(*[ResBlock(trunk_ch) for _ in range(n_blocks)])
        self.pool = nn.AdaptiveAvgPool2d(1)  # (B,C,1,1)

        head = []
        prev = trunk_ch
        for h in fc_hidden:
            head += [nn.Linear(prev, h), nn.ReLU(inplace=True)]
            if p_drop and p_drop > 0:
                head += [nn.Dropout(p_drop)]
            prev = h
        head += [nn.Linear(prev, 1)]
        self.head = nn.Sequential(*head)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).flatten(1)   # (B,C)
        return self.head(x)




In [8]:
import torch.nn as nn
import torch.nn.functional as F

class CenterWeightedHuber(nn.Module):
    def __init__(self, delta=150.0, clip=1500.0, k=400.0, min_w=0.5):
        super().__init__()
        self.delta = float(delta)
        self.clip  = float(clip)
        self.k     = float(k)
        self.min_w = float(min_w)

    def forward(self, pred_cp, true_cp):
        if pred_cp.dim() > 1 and pred_cp.size(-1) == 1:
            pred_cp = pred_cp.squeeze(-1)
        if true_cp.dim() > 1 and true_cp.size(-1) == 1:
            true_cp = true_cp.squeeze(-1)

        t = true_cp.clamp(-self.clip, self.clip)
        base = F.smooth_l1_loss(pred_cp, t, beta=self.delta, reduction='none')

        # emphasize near-equal positions
        w = torch.exp(- (t.abs()/self.k)**2)
        w = self.min_w + (1 - self.min_w) * w

        return (base * w).mean()



In [9]:
model = CNN(in_ch=in_ch, trunk_ch=TRUNK_CH, n_blocks=N_BLOCKS, fc_hidden=FC_HIDDEN, p_drop=P_DROP).to(device)

# -----------------------------
# Loss / Optim / Sched
# Target is y=tanh(cp/400) in [-1,1], so MSE is appropriate.
# -----------------------------
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.99),
    eps=1e-8
)

# Cosine LR is typically better than ReduceLROnPlateau for this setup
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-5
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

Nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {Nparams/1e6:.3f}M")



Params: 5.394M


In [10]:

model.train()
xb, yb = xb0.to(device), yb0.to(device)
with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
    pred = model(xb)
    loss = criterion(pred, yb)
print("one-step OK, loss:", float(loss))




one-step OK, loss: 0.6515698432922363


In [11]:
yb[0]

tensor([-0.9433], device='cuda:0')

In [12]:
# ---- Clean tiny overfit test (no dropout, no fancy loss) ----

from torch.utils.data import Subset, DataLoader

small_idx = list(range(min(4096, len(train_ds))))
tiny_loader = DataLoader(Subset(train_ds, small_idx), batch_size=256, shuffle=True)

model_small = CNN(in_ch=in_ch, trunk_ch=TRUNK_CH, n_blocks=4, fc_hidden=(128,), p_drop=0.0).to(device)
opt_small = torch.optim.Adam(model_small.parameters(), lr=5e-4)
crit_small = nn.MSELoss()

print("Starting tiny overfit test...")

for e in range(1500):   # 200–500 is enough
    tl = 0.0
    model_small.train()
    for xb, yb in tiny_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt_small.zero_grad(set_to_none=True)
        pred = model_small(xb)
        loss = crit_small(pred, yb)
        loss.backward()
        opt_small.step()
        tl += loss.item() * xb.size(0)
    print(f"(tiny) epoch {e+1:03d} loss {tl/len(small_idx):.6f}")



Starting tiny overfit test...
(tiny) epoch 001 loss 0.423865
(tiny) epoch 002 loss 0.309254
(tiny) epoch 003 loss 0.305500
(tiny) epoch 004 loss 0.291233
(tiny) epoch 005 loss 0.276728
(tiny) epoch 006 loss 0.268594
(tiny) epoch 007 loss 0.262467
(tiny) epoch 008 loss 0.243583
(tiny) epoch 009 loss 0.233724
(tiny) epoch 010 loss 0.229182
(tiny) epoch 011 loss 0.216561
(tiny) epoch 012 loss 0.188791
(tiny) epoch 013 loss 0.173947
(tiny) epoch 014 loss 0.154431
(tiny) epoch 015 loss 0.136759
(tiny) epoch 016 loss 0.128035
(tiny) epoch 017 loss 0.112886
(tiny) epoch 018 loss 0.119340
(tiny) epoch 019 loss 0.098125
(tiny) epoch 020 loss 0.080369
(tiny) epoch 021 loss 0.080626
(tiny) epoch 022 loss 0.074264
(tiny) epoch 023 loss 0.068037
(tiny) epoch 024 loss 0.056388
(tiny) epoch 025 loss 0.049622
(tiny) epoch 026 loss 0.048559
(tiny) epoch 027 loss 0.049267
(tiny) epoch 028 loss 0.053275
(tiny) epoch 029 loss 0.053396
(tiny) epoch 030 loss 0.041251
(tiny) epoch 031 loss 0.037121
(tiny) ep

In [13]:
# -----------------------------
# Train loop + early stopping
# -----------------------------
best_val = float('inf')
pat = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            pred = model(xb)
            loss = criterion(pred, yb)

        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * xb.size(0)


    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            pred = model(xb)
            val_loss += criterion(pred, yb).item() * xb.size(0)

    val_loss /= len(val_loader.dataset)
    scheduler.step()


    print(f"Epoch {epoch:02d} | train loss: {train_loss:.6f} | val loss: {val_loss:.6f}")

    if val_loss < best_val - 1e-6:
        best_val = val_loss
        pat = 0
        torch.save({
            "model_state": model.state_dict(),
            "input_dim": int(in_ch),
            "trunk_ch": int(TRUNK_CH),
            "n_blocks": int(N_BLOCKS),
            "fc_hidden": list(FC_HIDDEN),
            "p_drop": float(P_DROP),
        }, CKPT_PATH)



    else:
        pat += 1
        if pat >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val loss: {best_val:.6f}")
            break

print("Best val loss:", best_val)
print(f"Saved best model to: {CKPT_PATH}")

Epoch 01 | train loss: 0.183306 | val loss: 0.190431
Epoch 02 | train loss: 0.145528 | val loss: 0.167928
Epoch 03 | train loss: 0.127270 | val loss: 0.134840
Epoch 04 | train loss: 0.114611 | val loss: 0.112430
Epoch 05 | train loss: 0.106594 | val loss: 0.123285
Epoch 06 | train loss: 0.100424 | val loss: 0.105001
Epoch 07 | train loss: 0.094210 | val loss: 0.106576
Epoch 08 | train loss: 0.089809 | val loss: 0.116381
Epoch 09 | train loss: 0.085726 | val loss: 0.097140
Epoch 10 | train loss: 0.081706 | val loss: 0.091591
Epoch 11 | train loss: 0.077555 | val loss: 0.093257
Epoch 12 | train loss: 0.074573 | val loss: 0.089822
Epoch 13 | train loss: 0.070824 | val loss: 0.093077
Epoch 14 | train loss: 0.067761 | val loss: 0.089092
Epoch 15 | train loss: 0.064822 | val loss: 0.086220
Epoch 16 | train loss: 0.062221 | val loss: 0.085834
Epoch 17 | train loss: 0.059707 | val loss: 0.086310
Epoch 18 | train loss: 0.057183 | val loss: 0.084904
Epoch 19 | train loss: 0.054989 | val loss: 0.

In [ ]:
def evaluate_mse(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            pred = model(xb)
            total += criterion(pred, yb).item() * xb.size(0)
            n += xb.size(0)
    return total / n

final_val_mse = evaluate_mse(model, val_loader)
print("Final (reloaded) val MSE:", final_val_mse)

NameError: name 'model' is not defined

In [2]:
#not big enough to overfit
#need additional info
#look at tweaking params